# M1 — Lexical retrieval: E01–E03

**Goal.** Establish a reproducible high-recall lexical baseline for Avito service candidate generation, evaluated with the frozen `benchmark_aligned_proxy_v1` protocol from M0.

**Experiments.**

- **E01:** exact BM25 on title + parameters + description;
- **E02:** document-field ablations and adding search filters to the query;
- **E03:** character n-gram TF-IDF and reciprocal-rank fusion (RRF).

All retrieval uses the M0 category partition: `item_category_id == search_category`. The notebook logs only aggregate technical and quality metrics to a live ClearML task; it does not upload source texts or Parquet files.

## План ноутбука

1. **Reproducibility и ClearML** — загружаем локальную конфигурацию и создаём live Task без вывода секретов.
2. **Frozen proxy и category partition** — точно воспроизводим M0 split и допустимый corpus для каждого запроса.
3. **BM25 implementation** — проверяем sparse реализацию against `rank_bm25` перед вычислением полного corpus.
4. **E01** — строим базовый full-text BM25.
5. **E02** — меняем по одному текстовому полю или query representation.
6. **E03** — добавляем char TF-IDF и deterministic RRF fusion.
7. **Results и решение** — сравниваем macro Recall@50 и runtime, логируем агрегаты в ClearML и выбираем baseline.

Разделы ниже объясняют назначение следующих ячеек; комментарии в коде оставлены только для нетривиальных ограничений и решений.

## 1. Reproducibility and live ClearML

The notebook loads ClearML settings from the repository-local `.env` *before* importing ClearML. It requires a live configured task and deliberately has no offline fallback. No credential value is printed or persisted in notebook output.

In [1]:
from __future__ import annotations

import gc
import os
import re
import resource
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
DATA_DIR = Path(os.environ.get("AVITO_DATA_DIR", "/Users/kite/Downloads/dataset"))
TRAIN_PATH = DATA_DIR / "train.parquet"
BENCHMARK_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
BENCHMARK_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"

TOP_K = 200
METRIC_KS = (1, 5, 10, 20, 50, 200)
BM25_K1 = 1.5
BM25_B = 0.75
CHAR_MAX_FEATURES = 200_000

assert REPO_ROOT.joinpath(".env").exists(), "Create and fill .env before running M1."
assert all(path.exists() for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH))

NOTEBOOK_STARTED = time.perf_counter()


def load_dotenv(path: Path) -> None:
    """Minimal .env loader; never prints values and does not overwrite shell env."""
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


load_dotenv(REPO_ROOT / ".env")
required_clearml = ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY")
missing_clearml = [key for key in required_clearml if not os.environ.get(key)]
if missing_clearml:
    raise RuntimeError(f"Missing ClearML values in .env: {', '.join(missing_clearml)}")
if os.environ.get("CLEARML_OFFLINE_MODE", "").lower() in {"1", "true", "yes"}:
    raise RuntimeError("M1 requires a live ClearML task; CLEARML_OFFLINE_MODE must not be enabled.")

print({
    "seed": SEED,
    "data_dir": str(DATA_DIR),
    "top_k": TOP_K,
    "bm25": {"k1": BM25_K1, "b": BM25_B},
    "char_tfidf_max_features": CHAR_MAX_FEATURES,
    "clearml_credentials_present": True,
})

{'seed': 42, 'data_dir': '/Users/kite/Downloads/dataset', 'top_k': 200, 'bm25': {'k1': 1.5, 'b': 0.75}, 'char_tfidf_max_features': 200000, 'clearml_credentials_present': True}


In [2]:
# Import happens only after the .env credentials are available.
from clearml import Task

CLEARML_PROJECT = "avito-retrieval"
CLEARML_TASK_NAME = "E01-E03__lexical__bm25-tfidf__s42"

clearml_task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK_NAME,
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
clearml_task.connect(
    {
        "stage": "M1_lexical_retrieval",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "bm25_k1": BM25_K1,
        "bm25_b": BM25_B,
        "category_partition": "item_category_id == search_category; full-corpus fallback when no matching corpus partition exists",
        "char_tfidf": {"analyzer": "char_wb", "ngram_range": [3, 5], "max_features": CHAR_MAX_FEATURES},
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_project": CLEARML_PROJECT, "clearml_task_id": clearml_task.id, "offline_mode": False})

ClearML Task: created new task id=272e1172dc1b4aaaaa81d4f7d198ee3e


2026-09-18 11:06:23,631 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []


ClearML results page: https://app.clear.ml/projects/588424e922a44a95aa934ad17ca57931/tasks/272e1172dc1b4aaaaa81d4f7d198ee3e/output/log
2026-09-18 11:06:24,176 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


{'clearml_project': 'avito-retrieval', 'clearml_task_id': '272e1172dc1b4aaaaa81d4f7d198ee3e', 'offline_mode': False}


## 2. Recreate the frozen M0 proxy exactly

`query_group` is built from all search-side fields over the concatenation of train and benchmark query contexts. The split is then applied to the benchmark-corpus-aligned train positive rows, exactly as in M0.

In [3]:
SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]
ITEM_COLUMNS = [
    "item_id",
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
    "item_category_id",
]

load_started = time.perf_counter()
train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=SEARCH_COLUMNS)
benchmark_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS)
load_seconds = time.perf_counter() - load_started


def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column]
            .astype("string")
            .fillna("<NA>")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result

all_contexts = pd.concat(
    [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],
    ignore_index=True,
)
all_group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = all_group_ids[: len(train_pairs)]

benchmark_item_id_set = set(benchmark_items["item_id"].astype(str))
proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(benchmark_item_id_set)].copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
proxy_train_idx, proxy_valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))
proxy_train_pairs = proxy_pairs.iloc[proxy_train_idx].copy()
proxy_valid_pairs = proxy_pairs.iloc[proxy_valid_idx].copy()
assert set(proxy_train_pairs["query_group"]).isdisjoint(set(proxy_valid_pairs["query_group"]))

validation_queries = (
    proxy_valid_pairs.sort_values("query_group")
    .drop_duplicates("query_group")
    .loc[:, ["query_group", *SEARCH_COLUMNS]]
    .reset_index(drop=True)
)
gold_by_group = (
    proxy_valid_pairs.groupby("query_group", sort=False)["item_id"]
    .agg(lambda values: frozenset(values.astype(str)))
    .to_dict()
)
gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]

proxy_summary = pd.DataFrame(
    [
        {"partition": "train", "positive_rows": len(proxy_train_pairs), "query_groups": proxy_train_pairs["query_group"].nunique()},
        {"partition": "validation", "positive_rows": len(proxy_valid_pairs), "query_groups": proxy_valid_pairs["query_group"].nunique()},
    ]
)
display(proxy_summary)
print({
    "load_seconds": round(load_seconds, 2),
    "proxy_positive_rows": len(proxy_pairs),
    "validation_unique_queries": len(validation_queries),
    "validation_gold_cardinality_median": float(np.median([len(gold) for gold in gold_sets])),
})

,partition,positive_rows,query_groups
0,train,26377,21239
1,validation,6633,5310


{'load_seconds': 1.14, 'proxy_positive_rows': 33010, 'validation_unique_queries': 5310, 'validation_gold_cardinality_median': 1.0}


In [4]:
# M0 hard partition, with the documented fallback if a category has no corpus rows.
# In this split, 5,309 queries have category 114; one query has category 0,
# which has no matching item_category_id and therefore uses the full corpus.
candidate_items = benchmark_items.reset_index(drop=True)
candidate_item_ids = candidate_items["item_id"].astype(str).to_numpy()
all_candidate_indices = np.arange(len(candidate_items), dtype=np.int64)
category_to_indices = {
    category: group.index.to_numpy(dtype=np.int64)
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}


def allowed_indices_for_category(category: object) -> tuple[np.ndarray, bool]:
    indices = category_to_indices.get(category)
    if indices is None or len(indices) == 0:
        return all_candidate_indices, True
    return indices, False


allowed_indices_by_query = []
used_full_corpus_fallback = []
for category in validation_queries["search_category"]:
    allowed, fallback = allowed_indices_for_category(category)
    allowed_indices_by_query.append(allowed)
    used_full_corpus_fallback.append(fallback)

category_oracle_recall = float(
    np.mean([
        len(gold & set(candidate_item_ids[allowed])) / len(gold)
        for gold, allowed in zip(gold_sets, allowed_indices_by_query, strict=True)
    ])
)
assert category_oracle_recall == 1.0, "The category rule plus fallback removes a validation positive."

print({
    "validation_categories": validation_queries["search_category"].value_counts().to_dict(),
    "items_in_category_114": int((candidate_items["item_category_id"] == 114).sum()),
    "full_benchmark_items": len(candidate_items),
    "full_corpus_fallback_queries": int(sum(used_full_corpus_fallback)),
    "validation_category_oracle_recall@50": category_oracle_recall,
})

{'validation_categories': {114: 5309, 0: 1}, 'items_in_category_114': 187336, 'full_benchmark_items': 189212, 'full_corpus_fallback_queries': 1, 'validation_category_oracle_recall@50': 1.0}


## 3. Exact sparse BM25 implementation

`rank_bm25` is used as a reference on a small deterministic example. The full corpus uses an equivalent sparse CSC implementation to avoid storing hundreds of thousands of Python token-frequency dictionaries. It applies standard Okapi BM25 with the same negative-IDF handling as `rank_bm25.BM25Okapi`.

In [5]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")


def normalize_russian_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.lower().replace("ё", "е")
    text = NON_WORD_RE.sub(" ", text)
    return " ".join(text.split())


class SparseBM25:
    """Memory-efficient exact Okapi BM25 over a CountVectorizer CSC matrix."""

    def __init__(self, k1: float = 1.5, b: float = 0.75, epsilon: float = 0.25):
        self.k1 = k1
        self.b = b
        self.epsilon = epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        self.vectorizer = CountVectorizer(
            preprocessor=normalize_russian_text,
            token_pattern=TOKEN_PATTERN,
            lowercase=False,
            dtype=np.float32,
        )
        counts_csr = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts_csr.sum(axis=1)).ravel().astype(np.float32)
        self.n_docs = counts_csr.shape[0]
        self.avgdl = float(self.doc_len.mean())
        self.matrix = counts_csr.tocsc()
        del counts_csr

        document_frequency = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - document_frequency + 0.5) / (document_frequency + 0.5))
        average_idf = float(idf.mean())
        idf[idf < 0] = self.epsilon * average_idf
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1.0 - self.b + self.b * self.doc_len / self.avgdl)
        self.analyzer = self.vectorizer.build_analyzer()
        return self

    @property
    def n_features(self) -> int:
        return len(self.vectorizer.vocabulary_)

    def score(self, query: str) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(self.analyzer(query)).items():
            feature_idx = self.vectorizer.vocabulary_.get(token)
            if feature_idx is None:
                continue
            start, stop = self.matrix.indptr[feature_idx : feature_idx + 2]
            rows = self.matrix.indices[start:stop]
            term_tf = self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature_idx] * (
                term_tf * (self.k1 + 1.0) / (term_tf + self.norm[rows])
            )
        return scores

    def top_k(self, query: str, k: int, allowed_indices: np.ndarray | None = None) -> np.ndarray:
        scores = self.score(query)
        allowed = np.arange(self.n_docs, dtype=np.int64) if allowed_indices is None else allowed_indices
        k = min(k, len(allowed))
        eligible_scores = scores[allowed]
        selected_positions = np.argpartition(eligible_scores, len(allowed) - k)[len(allowed) - k :]
        selected_positions = selected_positions[np.argsort(eligible_scores[selected_positions])[::-1]]
        return allowed[selected_positions]


# Correctness check against the open-source reference implementation.
toy_documents = ["ремонт ноутбука", "ремонт телевизора", "установка двери"]
toy_query = "ремонт"
reference_scores = BM25Okapi([document.split() for document in toy_documents], k1=BM25_K1, b=BM25_B).get_scores(toy_query.split())
checked_scores = SparseBM25(k1=BM25_K1, b=BM25_B).fit(toy_documents).score(toy_query)
np.testing.assert_allclose(checked_scores, reference_scores, rtol=1e-6, atol=1e-6)
print("SparseBM25 agrees with rank_bm25.BM25Okapi on the deterministic sanity check.")

SparseBM25 agrees with rank_bm25.BM25Okapi on the deterministic sanity check.


In [6]:
def compose_document_text(frame: pd.DataFrame, fields: tuple[str, ...]) -> list[str]:
    text = frame[fields[0]].fillna("").astype(str)
    for field in fields[1:]:
        text = text.str.cat(frame[field].fillna("").astype(str), sep=" ")
    return text.tolist()


def retrieve_bm25(index: SparseBM25, queries: list[str], allowed_by_query: list[np.ndarray], top_k: int) -> tuple[list[np.ndarray], float]:
    started = time.perf_counter()
    rankings = [
        index.top_k(query, top_k, allowed)
        for query, allowed in zip(queries, allowed_by_query, strict=True)
    ]
    return rankings, time.perf_counter() - started


def top_k_from_scores(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    selected = np.argpartition(scores, len(scores) - k)[len(scores) - k :]
    return selected[np.argsort(scores[selected])[::-1]]


def macro_recall(rankings: list[np.ndarray], gold: list[frozenset[str]], k: int) -> float:
    recalls = []
    for ranking, relevant in zip(rankings, gold, strict=True):
        predicted = set(candidate_item_ids[ranking[:k]])
        recalls.append(len(predicted & relevant) / len(relevant))
    return float(np.mean(recalls))


def hit_rate(rankings: list[np.ndarray], gold: list[frozenset[str]], k: int) -> float:
    hits = []
    for ranking, relevant in zip(rankings, gold, strict=True):
        hits.append(bool(set(candidate_item_ids[ranking[:k]]) & relevant))
    return float(np.mean(hits))


def peak_rss_mb() -> float:
    max_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return float(max_rss / (1024 * 1024 if sys.platform == "darwin" else 1024))


def evaluate(experiment: str, rankings: list[np.ndarray], *, index_seconds: float, retrieval_seconds: float,
             n_features: int, representation: str) -> dict[str, object]:
    record: dict[str, object] = {
        "experiment": experiment,
        "representation": representation,
        "index_seconds": index_seconds,
        "retrieval_seconds": retrieval_seconds,
        "total_seconds": index_seconds + retrieval_seconds,
        "features": n_features,
        "peak_rss_mb": peak_rss_mb(),
    }
    for k in METRIC_KS:
        record[f"recall@{k}"] = macro_recall(rankings, gold_sets, k)
    record["hit_rate@50"] = hit_rate(rankings, gold_sets, 50)
    return record


def fit_sparse_bm25(documents: list[str]) -> tuple[SparseBM25, float]:
    started = time.perf_counter()
    index = SparseBM25(k1=BM25_K1, b=BM25_B).fit(documents)
    return index, time.perf_counter() - started


def rrf_fuse(rankings_by_source: list[np.ndarray], top_k: int, rrf_k: int = 60) -> np.ndarray:
    scores: dict[int, float] = {}
    for rankings in rankings_by_source:
        for rank, item_idx in enumerate(rankings, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (rrf_k + rank)
    ordered = sorted(scores, key=lambda item_idx: (-scores[item_idx], item_idx))
    return np.asarray(ordered[:top_k], dtype=np.int64)


query_text = validation_queries["search_query"].fillna("").astype(str).tolist()
query_with_filters = (
    validation_queries["search_query"].fillna("").astype(str)
    + " "
    + validation_queries["search_infm_params_text"].fillna("").astype(str)
).tolist()

assert len(query_text) == len(gold_sets) == 5310
results: list[dict[str, object]] = []
print({"validation_queries": len(query_text), "retrieval_top_k": TOP_K})

{'validation_queries': 5310, 'retrieval_top_k': 200}


## 4. E01 — Plain BM25

The prescribed first baseline indexes the concatenation of title, parameters and description; the query contains only `search_query`.

In [7]:
all_fields = ("item_title_raw", "item_infm_params_text", "item_description_raw")
all_documents = compose_document_text(candidate_items, all_fields)
all_bm25, all_index_seconds = fit_sparse_bm25(all_documents)
rankings_all, all_retrieval_seconds = retrieve_bm25(all_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E01_bm25_title_params_description__query",
    rankings_all,
    index_seconds=all_index_seconds,
    retrieval_seconds=all_retrieval_seconds,
    n_features=all_bm25.n_features,
    representation="title + parameters + description | query",
))
print(results[-1])

{'experiment': 'E01_bm25_title_params_description__query', 'representation': 'title + parameters + description | query', 'index_seconds': 44.64419783300036, 'retrieval_seconds': 6.0552068750002945, 'total_seconds': 50.69940470800066, 'features': 595180, 'peak_rss_mb': 1963.234375, 'recall@1': 0.019350282485875708, 'recall@5': 0.07426867545511613, 'recall@10': 0.11671971123666039, 'recall@20': 0.1679728126027561, 'recall@50': 0.2840713388933728, 'recall@200': 0.5145393238274594, 'hit_rate@50': 0.29397363465160076}


## 5. E02 — Controlled lexical ablations

Only one textual representation changes at a time. The all-fields index is reused for the query-filter ablation, so its retrieval timing excludes a duplicated index build.

In [8]:
# E02a: title only.
title_documents = compose_document_text(candidate_items, ("item_title_raw",))
title_bm25, title_index_seconds = fit_sparse_bm25(title_documents)
rankings_title, title_retrieval_seconds = retrieve_bm25(title_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02a_bm25_title__query",
    rankings_title,
    index_seconds=title_index_seconds,
    retrieval_seconds=title_retrieval_seconds,
    n_features=title_bm25.n_features,
    representation="title | query",
))
print(results[-1])
del title_bm25, title_documents
gc.collect()

# E02b: title + item parameters.
title_params_documents = compose_document_text(candidate_items, ("item_title_raw", "item_infm_params_text"))
title_params_bm25, title_params_index_seconds = fit_sparse_bm25(title_params_documents)
rankings_title_params, title_params_retrieval_seconds = retrieve_bm25(title_params_bm25, query_text, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02b_bm25_title_params__query",
    rankings_title_params,
    index_seconds=title_params_index_seconds,
    retrieval_seconds=title_params_retrieval_seconds,
    n_features=title_params_bm25.n_features,
    representation="title + parameters | query",
))
print(results[-1])
del title_params_bm25, title_params_documents
gc.collect()

# E02c: add the search filter text to the E01 query representation.
rankings_all_filters, all_filters_retrieval_seconds = retrieve_bm25(all_bm25, query_with_filters, allowed_indices_by_query, TOP_K)
results.append(evaluate(
    "E02c_bm25_title_params_description__query_filters",
    rankings_all_filters,
    index_seconds=0.0,
    retrieval_seconds=all_filters_retrieval_seconds,
    n_features=all_bm25.n_features,
    representation="title + parameters + description | query + search filters (reused E01 index)",
))
print(results[-1])

{'experiment': 'E02a_bm25_title__query', 'representation': 'title | query', 'index_seconds': 0.7660522499973013, 'retrieval_seconds': 4.218788458001654, 'total_seconds': 4.984840707998956, 'features': 42417, 'peak_rss_mb': 1963.234375, 'recall@1': 0.015819209039548022, 'recall@5': 0.06155681104833648, 'recall@10': 0.09379912115505337, 'recall@20': 0.13725038113173707, 'recall@50': 0.22069043135144834, 'recall@200': 0.3856454279137895, 'hit_rate@50': 0.23107344632768362}


{'experiment': 'E02b_bm25_title_params__query', 'representation': 'title + parameters | query', 'index_seconds': 15.02130683300129, 'retrieval_seconds': 5.385957749997033, 'total_seconds': 20.407264582998323, 'features': 126289, 'peak_rss_mb': 1963.234375, 'recall@1': 0.01820464532328939, 'recall@5': 0.05619899560577526, 'recall@10': 0.09064344005021972, 'recall@20': 0.13950168893671716, 'recall@50': 0.21605536125310135, 'recall@200': 0.3954993274145816, 'hit_rate@50': 0.22523540489642185}


ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


{'experiment': 'E02c_bm25_title_params_description__query_filters', 'representation': 'title + parameters + description | query + search filters (reused E01 index)', 'index_seconds': 0.0, 'retrieval_seconds': 20.42625825000141, 'total_seconds': 20.42625825000141, 'features': 595180, 'peak_rss_mb': 2041.5625, 'recall@1': 0.01977401129943503, 'recall@5': 0.07343691148775894, 'recall@10': 0.11187696170747018, 'recall@20': 0.1684180790960452, 'recall@50': 0.278385570800825, 'recall@200': 0.5055377694078259, 'hit_rate@50': 0.2887005649717514}


## 6. E03 — Strong lexical alternatives and fusion

The char experiment uses short titles only: it tests typo-tolerant lexical matching without creating an impractically large n-gram index over the long parameter texts. RRF uses no relevance labels and selects 200 candidates before Recall@K diagnostics are calculated.

In [9]:
# E03a: char n-gram TF-IDF on short titles.
char_documents = compose_document_text(candidate_items, ("item_title_raw",))
char_fit_started = time.perf_counter()
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    preprocessor=normalize_russian_text,
    lowercase=False,
    ngram_range=(3, 5),
    min_df=2,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    dtype=np.float32,
)
char_matrix = char_vectorizer.fit_transform(char_documents)
char_index_seconds = time.perf_counter() - char_fit_started

char_retrieval_started = time.perf_counter()
rankings_char = []
for query, allowed in zip(query_text, allowed_indices_by_query, strict=True):
    query_vector = char_vectorizer.transform([query])
    scores = (query_vector @ char_matrix.T).toarray().ravel()
    eligible_scores = scores[allowed]
    selected_positions = top_k_from_scores(eligible_scores, TOP_K)
    rankings_char.append(allowed[selected_positions])
char_retrieval_seconds = time.perf_counter() - char_retrieval_started
results.append(evaluate(
    "E03a_char_tfidf_title__query",
    rankings_char,
    index_seconds=char_index_seconds,
    retrieval_seconds=char_retrieval_seconds,
    n_features=len(char_vectorizer.vocabulary_),
    representation="char_wb 3–5 grams: title | query",
))
print(results[-1])

# E03b: reciprocal-rank fusion of title-only and E01 all-fields BM25.
rrf_bm25_started = time.perf_counter()
rankings_rrf_bm25 = [rrf_fuse([title_ranking, all_ranking], TOP_K) for title_ranking, all_ranking in zip(rankings_title, rankings_all, strict=True)]
rrf_bm25_seconds = time.perf_counter() - rrf_bm25_started
results.append(evaluate(
    "E03b_rrf_bm25_title_all_fields",
    rankings_rrf_bm25,
    index_seconds=0.0,
    retrieval_seconds=rrf_bm25_seconds,
    n_features=0,
    representation="RRF(title-BM25, all-fields-BM25); reused indices",
))
print(results[-1])

# E03c: fuse the strongest broad BM25 source with char TF-IDF.
rrf_char_started = time.perf_counter()
rankings_rrf_char = [rrf_fuse([all_ranking, char_ranking], TOP_K) for all_ranking, char_ranking in zip(rankings_all, rankings_char, strict=True)]
rrf_char_seconds = time.perf_counter() - rrf_char_started
results.append(evaluate(
    "E03c_rrf_bm25_all_fields_char_tfidf",
    rankings_rrf_char,
    index_seconds=0.0,
    retrieval_seconds=rrf_char_seconds,
    n_features=0,
    representation="RRF(all-fields BM25, title-char TF-IDF); reused indices",
))
print(results[-1])

{'experiment': 'E03a_char_tfidf_title__query', 'representation': 'char_wb 3–5 grams: title | query', 'index_seconds': 5.105183042000135, 'retrieval_seconds': 318.66400287500073, 'total_seconds': 323.76918591700087, 'features': 97143, 'peak_rss_mb': 2041.5625, 'recall@1': 0.017608286252354048, 'recall@5': 0.058857501569365984, 'recall@10': 0.09303515379786566, 'recall@20': 0.1392467043314501, 'recall@50': 0.2364576271186441, 'recall@200': 0.43353261291961853, 'hit_rate@50': 0.24538606403013183}


{'experiment': 'E03b_rrf_bm25_title_all_fields', 'representation': 'RRF(title-BM25, all-fields-BM25); reused indices', 'index_seconds': 0.0, 'retrieval_seconds': 0.661468166999839, 'total_seconds': 0.661468166999839, 'features': 0, 'peak_rss_mb': 2041.5625, 'recall@1': 0.020433145009416197, 'recall@5': 0.07228311362209668, 'recall@10': 0.11021155053358443, 'recall@20': 0.1688932831136221, 'recall@50': 0.283104370310585, 'recall@200': 0.5179554300062774, 'hit_rate@50': 0.29453860640301316}


{'experiment': 'E03c_rrf_bm25_all_fields_char_tfidf', 'representation': 'RRF(all-fields BM25, title-char TF-IDF); reused indices', 'index_seconds': 0.0, 'retrieval_seconds': 0.6365716670006805, 'total_seconds': 0.6365716670006805, 'features': 0, 'peak_rss_mb': 2041.5625, 'recall@1': 0.022598870056497175, 'recall@5': 0.07620841180163214, 'recall@10': 0.11653797865662271, 'recall@20': 0.18228679042238363, 'recall@50': 0.2972636684303351, 'recall@200': 0.5344497354497354, 'hit_rate@50': 0.30847457627118646}


## 7. Results, ClearML logging and decision

The table is ordered by the frozen primary metric, macro Recall@50. Index time and retrieval time are reported separately; reused-index variants have `index_seconds = 0` by design. The ClearML task receives aggregate metrics only.

In [10]:
results_frame = pd.DataFrame(results).sort_values("recall@50", ascending=False).reset_index(drop=True)
display(results_frame)

best_result = results_frame.iloc[0].to_dict()
notebook_seconds = time.perf_counter() - NOTEBOOK_STARTED

for _, result in results_frame.iterrows():
    experiment = str(result["experiment"])
    for k in METRIC_KS:
        clearml_logger.report_scalar(
            title=f"Recall@{k}", series=experiment, value=float(result[f"recall@{k}"]), iteration=0
        )
    clearml_logger.report_scalar(title="HitRate@50", series=experiment, value=float(result["hit_rate@50"]), iteration=0)
    clearml_logger.report_scalar(title="Index seconds", series=experiment, value=float(result["index_seconds"]), iteration=0)
    clearml_logger.report_scalar(title="Retrieval seconds", series=experiment, value=float(result["retrieval_seconds"]), iteration=0)
    clearml_logger.report_scalar(title="Peak RSS MB", series=experiment, value=float(result["peak_rss_mb"]), iteration=0)

clearml_logger.report_scalar(title="M1 wall time seconds", series="notebook", value=float(notebook_seconds), iteration=0)
clearml_logger.report_scalar(title="Category-114 partition items", series="corpus", value=float((candidate_items["item_category_id"] == 114).sum()), iteration=0)
try:
    clearml_logger.report_table(
        title="M1 validation summary",
        series="E01-E03",
        iteration=0,
        table_plot=results_frame,
    )
    table_log_status = "logged"
except Exception as exc:
    table_log_status = f"table logging skipped: {type(exc).__name__}"

clearml_task.set_parameter("results/best_experiment", str(best_result["experiment"]))
clearml_task.set_parameter("results/best_recall_at_50", float(best_result["recall@50"]))
clearml_task.set_parameter("results/notebook_seconds", float(notebook_seconds))
clearml_task.close()

print({
    "best_experiment": best_result["experiment"],
    "best_recall@50": round(float(best_result["recall@50"]), 6),
    "notebook_seconds": round(notebook_seconds, 2),
    "clearml_table_status": table_log_status,
})

,experiment,representation,index_seconds,retrieval_seconds,total_seconds,features,peak_rss_mb,recall@1,recall@5,recall@10,recall@20,recall@50,recall@200,hit_rate@50
0,E03c_rrf_bm25_all_fields_char_tfidf,"RRF(all-fields BM25, title-char TF-IDF); reuse...",0.000000,0.636572,0.636572,0,2041.562500,0.022599,0.076208,0.116538,0.182287,0.297264,0.534450,0.308475
1,E01_bm25_title_params_description__query,title + parameters + description | query,44.644198,6.055207,50.699405,595180,1963.234375,0.019350,0.074269,0.116720,0.167973,0.284071,0.514539,0.293974
2,E03b_rrf_bm25_title_all_fields,"RRF(title-BM25, all-fields-BM25); reused indices",0.000000,0.661468,0.661468,0,2041.562500,0.020433,0.072283,0.110212,0.168893,0.283104,0.517955,0.294539
3,E02c_bm25_title_params_description__query_filters,title + parameters + description | query + sea...,0.000000,20.426258,20.426258,595180,2041.562500,0.019774,0.073437,0.111877,0.168418,0.278386,0.505538,0.288701
4,E03a_char_tfidf_title__query,char_wb 3–5 grams: title | query,5.105183,318.664003,323.769186,97143,2041.562500,0.017608,0.058858,0.093035,0.139247,0.236458,0.433533,0.245386
5,E02a_bm25_title__query,title | query,0.766052,4.218788,4.984841,42417,1963.234375,0.015819,0.061557,0.093799,0.137250,0.220690,0.385645,0.231073
6,E02b_bm25_title_params__query,title + parameters | query,15.021307,5.385958,20.407265,126289,1963.234375,0.018205,0.056199,0.090643,0.139502,0.216055,0.395499,0.225235


{'best_experiment': 'E03c_rrf_bm25_all_fields_char_tfidf', 'best_recall@50': 0.297264, 'notebook_seconds': 533.55, 'clearml_table_status': 'logged'}


## M1 conclusion

Use the best row in the executed results table as the retained lexical baseline. Before replacing it in a later stage, compare any new candidate source on the same frozen split and report source-union Recall@50.

**Scope note.** The frozen validation fold has 5,309 queries with `search_category = 114` and one query with category `0`. Category 114 contains 187,336 of 189,212 corpus items, so the hard partition preserves known positives but gives little speed-up for the dominant category. The char-TF-IDF comparison intentionally indexes titles only; applying char n-grams to the long parameter field exceeded the one-hour cell limit and is not retained as a feasible M1 baseline. Category 0 has no matching corpus partition and correctly triggers the documented full-corpus fallback.